**Title:**  Project 1 — SBA Loan Default Prediction     
**Name:** Hima Varsha Komanduri     
**NetID:** dal850892        
**Course:** BUAN 6341 (Spring 2026)     
**Models:** sklearn Logistic Regression + H2O-3 GLM (binomial)      
**Tuning metric:** AUCPR or Log-loss        
**Final reporting on Test:** AUC, AUCPR, Log-loss, confusion matrix @ F1-opt threshold      
**Scoring:** Separate file using scoring_template.py        

## 1. Data Loading and Initial Audit

Load the SBA loan dataset and perform an initial audit.  
This includes inspecting dataset shape, target distribution, missing values, data types, and identifying a unique identifier column for scoring.

In [62]:
import pandas as pd
import numpy as np

train_path = "./data/SBA_loans_project_1.csv.zip"
holdout_path = "./data/SBA_loans_project_1_holdout_students_valid_no_labels.csv.zip"

train = pd.read_csv(train_path)
holdout = pd.read_csv(holdout_path)

# Target checks
print("\nMIS_Status counts (train):")
print(train["MIS_Status"].value_counts(dropna=False))

# Missingness checks
na_top15 = train.isna().sum().sort_values(ascending=False).head(15)
print("\nTop 15 columns by missing values (train):")
print(na_top15)

# Dtypes checks
print("\nDtypes summary (train):")
print(train.dtypes.value_counts())

# Duplicates
print("\nDuplicate rows in train:", train.duplicated().sum())

# Candidate ID column for scoring output 'index'
candidate_ids = ["LoanNr_ChkDgt", "LoanNumber", "ID", "Id", "loan_id"]
id_col = next((c for c in candidate_ids if c in holdout.columns), None)

print("\nChosen id_col:", id_col)
if id_col is not None:
    print("Unique in train?:", train[id_col].is_unique)
    print("Unique in holdout?:", holdout[id_col].is_unique)
    print("Missing in holdout?:", int(holdout[id_col].isna().sum()))

print("\nMIS_Status in holdout columns?:", "MIS_Status" in holdout.columns)


MIS_Status counts (train):
MIS_Status
0    667366
1    141881
Name: count, dtype: int64

Top 15 columns by missing values (train):
RevLineCr            4078
LowDoc               2320
BankState            1413
Bank                 1406
NewExist              124
City                   27
State                  12
index                   0
SBA_Appv                0
GrAppv                  0
BalanceGross            0
DisbursementGross       0
RetainedJob             0
UrbanRural              0
FranchiseCode           0
dtype: int64

Dtypes summary (train):
int64      9
object     6
float64    5
Name: count, dtype: int64

Duplicate rows in train: 0

Chosen id_col: None

MIS_Status in holdout columns?: False


In [63]:
# Shapes 
print(train.shape)
print(holdout.shape)

(809247, 20)
(89917, 19)


In [64]:
#Target Distribution
train["MIS_Status"].value_counts()
train["MIS_Status"].value_counts(normalize=True) * 100

MIS_Status
0    82.467528
1    17.532472
Name: proportion, dtype: float64

In [65]:
#Column Inspection
print(train.columns.tolist())
print(holdout.columns.tolist())

['index', 'City', 'State', 'Zip', 'Bank', 'BankState', 'NAICS', 'NoEmp', 'NewExist', 'CreateJob', 'RetainedJob', 'FranchiseCode', 'UrbanRural', 'RevLineCr', 'LowDoc', 'DisbursementGross', 'BalanceGross', 'GrAppv', 'SBA_Appv', 'MIS_Status']
['index', 'City', 'State', 'Zip', 'Bank', 'BankState', 'NAICS', 'NoEmp', 'NewExist', 'CreateJob', 'RetainedJob', 'FranchiseCode', 'UrbanRural', 'RevLineCr', 'LowDoc', 'DisbursementGross', 'BalanceGross', 'GrAppv', 'SBA_Appv']


In [66]:
# ID uniqueness check

print("Unique in train:", train["index"].is_unique)
print("Unique in holdout:", holdout["index"].is_unique)

Unique in train: True
Unique in holdout: True


### Unique Identifier

The column `index` uniquely identifies each loan record and will be used as the identifier in the scoring output.

## 2. Data Cleaning
Handle missing values and ensuring consistent data types before feature engineering and modeling

In [67]:
# Make a working copy
df = train.copy()

# Fill categorical missing values
cat_fill_unknown = ["RevLineCr", "LowDoc", "BankState", "Bank", "City", "State"]
for col in cat_fill_unknown:
    df[col] = df[col].fillna("Unknown")

# Fill numeric / ordinal
df["NewExist"] = df["NewExist"].fillna(df["NewExist"].mode()[0])

# Verify missing values
print(df.isna().sum())

index                0
City                 0
State                0
Zip                  0
Bank                 0
BankState            0
NAICS                0
NoEmp                0
NewExist             0
CreateJob            0
RetainedJob          0
FranchiseCode        0
UrbanRural           0
RevLineCr            0
LowDoc               0
DisbursementGross    0
BalanceGross         0
GrAppv               0
SBA_Appv             0
MIS_Status           0
dtype: int64


### Missing Value Handling

Several categorical variables contained missing values, including `RevLineCr`, `LowDoc`, `BankState`, `Bank`, `City`, and `State`.  
These were filled with the category `"Unknown"` to preserve information while avoiding row deletion.

The column `NewExist`, which represents whether the business is new or existing, contained a small number of missing values.  
These were imputed using the mode of the column.

After applying these steps, the dataset contained no remaining missing values.

## 3. Feature Engineering

Create additional features to capture meaningful relationships within the loan and business characteristics.
These features aim to improve the predictive power of the linear models.

In [68]:
# Feature Engineering

df["loan_to_sba_ratio"] = df["SBA_Appv"] / (df["GrAppv"] + 1)

df["disbursement_to_approval"] = df["DisbursementGross"] / (df["GrAppv"] + 1)

df["balance_to_disbursement"] = df["BalanceGross"] / (df["DisbursementGross"] + 1)

df["jobs_created_ratio"] = df["CreateJob"] / (df["NoEmp"] + 1)

df["jobs_retained_ratio"] = df["RetainedJob"] / (df["NoEmp"] + 1)

df["total_jobs"] = df["CreateJob"] + df["RetainedJob"]

df["jobs_per_employee"] = (df["CreateJob"] + df["RetainedJob"]) / (df["NoEmp"] + 1)

df["is_urban"] = (df["UrbanRural"] == 1).astype(int)

df["is_franchise"] = (df["FranchiseCode"] > 0).astype(int)

df["sba_guarantee_diff"] = df["GrAppv"] - df["SBA_Appv"]

df["log_approval"] = np.log1p(df["GrAppv"])

In [69]:
#Verification
new_features = [
"loan_to_sba_ratio",
"disbursement_to_approval",
"balance_to_disbursement",
"jobs_created_ratio",
"jobs_retained_ratio",
"total_jobs",
"jobs_per_employee",
"is_urban",
"is_franchise",
"sba_guarantee_diff",
"log_approval"
]

print("Number of engineered features:", len(new_features))

Number of engineered features: 11


## 4. Train / Validation / Test Split

Split the dataset into training, validation, and test sets.

Training data is used to train the models.  
Validation data is used for hyperparameter tuning and model comparison.  
Test data is held out and used only for the final evaluation of model performance.

In [70]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["MIS_Status"])
y = df["MIS_Status"]

# First split: Train vs Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    stratify=y,
    random_state=42
)

# Second split: Validation vs Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (485548, 30)
Validation shape: (161849, 30)
Test shape: (161850, 30)


In [71]:
#Distribution check
print("Train distribution")
print(y_train.value_counts(normalize=True))

print("\nValidation distribution")
print(y_val.value_counts(normalize=True))

print("\nTest distribution")
print(y_test.value_counts(normalize=True))

Train distribution
MIS_Status
0    0.824674
1    0.175326
Name: proportion, dtype: float64

Validation distribution
MIS_Status
0    0.824676
1    0.175324
Name: proportion, dtype: float64

Test distribution
MIS_Status
0    0.824677
1    0.175323
Name: proportion, dtype: float64


### Class Distribution After Split

Stratified sampling was used to maintain the original class distribution across the training, validation, and test datasets.

The proportion of the majority class (0) is approximately 82%, while the minority class (1) represents approximately 18% in all splits. This ensures consistent evaluation and prevents sampling bias.

## 5. Categorical Feature Processing
### 5.1 Categorical Feature Inspection
Before applying categorical encoding, inspect categorical variables and their cardinality (number of unique values).  
This helps determine the appropriate encoding method.

In [72]:
#Inspection
cat_cols = X_train.select_dtypes(include="object").columns.tolist()

print("Categorical columns:")
print(cat_cols)

print("\nUnique values per categorical column:")
for col in cat_cols:
    print(col, ":", X_train[col].nunique())

Categorical columns:
['City', 'State', 'Bank', 'BankState', 'RevLineCr', 'LowDoc']

Unique values per categorical column:
City : 25474
State : 52
Bank : 5207
BankState : 57
RevLineCr : 17
LowDoc : 9


### 5.2 Categorical Encoding

Categorical variables are encoded using methods appropriate to their cardinality.

Low-cardinality variables are encoded using one-hot encoding,       
moderate-cardinality variables using WOE encoding, and      
high-cardinality variables using target encoding.

This approach prevents excessive dimensionality while preserving predictive information.

In [73]:
import category_encoders as ce

# Define columns
onehot_cols = ["RevLineCr", "LowDoc"]          # small-cardinality
woe_cols = ["State", "BankState"]             # moderate-cardinality
target_cols = ["City", "Bank"]                # high-cardinality

# 1) One-hot encode low-cardinality 
X_train_oh = pd.get_dummies(X_train, columns=onehot_cols, drop_first=True)
X_val_oh   = pd.get_dummies(X_val,   columns=onehot_cols, drop_first=True)
X_test_oh  = pd.get_dummies(X_test,  columns=onehot_cols, drop_first=True)

# ALIGN val/test columns to train columns 
X_val_oh  = X_val_oh.reindex(columns=X_train_oh.columns, fill_value=0)
X_test_oh = X_test_oh.reindex(columns=X_train_oh.columns, fill_value=0)

# 2) Target encode high-cardinality 
target_encoder = ce.TargetEncoder(cols=target_cols)
target_encoder.fit(X_train_oh, y_train)

X_train_te = target_encoder.transform(X_train_oh)
X_val_te   = target_encoder.transform(X_val_oh)
X_test_te  = target_encoder.transform(X_test_oh)

# 3) WOE encode moderate-cardinality 
woe_encoder = ce.WOEEncoder(cols=woe_cols)
woe_encoder.fit(X_train_te, y_train)

X_train_enc = woe_encoder.transform(X_train_te)
X_val_enc   = woe_encoder.transform(X_val_te)
X_test_enc  = woe_encoder.transform(X_test_te)

print("Train shape after encoding:", X_train_enc.shape)
print("Validation shape after encoding:", X_val_enc.shape)
print("Test shape after encoding:", X_test_enc.shape)

Train shape after encoding: (485548, 52)
Validation shape after encoding: (161849, 52)
Test shape after encoding: (161850, 52)


In [74]:
#Sanity checks
# Sanity checks after encoding
print("Any NA in train encoded?", X_train_enc.isna().any().any())
print("Any NA in val encoded?", X_val_enc.isna().any().any())
print("Any NA in test encoded?", X_test_enc.isna().any().any())

print("\nAll shapes equal columns?", 
      X_train_enc.shape[1] == X_val_enc.shape[1] == X_test_enc.shape[1])

Any NA in train encoded? False
Any NA in val encoded? False
Any NA in test encoded? False

All shapes equal columns? True


### Results of Categorical Encoding
After applying the encoding procedures, the categorical variables were successfully transformed into numerical features suitable for machine learning models.

The following encoding methods were applied:

• One-hot encoding for low-cardinality variables (`RevLineCr`, `LowDoc`)

• Target encoding for high-cardinality variables (`City`, `Bank`)

• Weight-of-evidence (WOE) encoding for moderate-cardinality variables (`State`, `BankState`)

All encoders were trained using only the training dataset to prevent data leakage, and the same transformations were applied to the validation and test datasets.

After encoding, the resulting feature matrices contain **52 features**, and the training, validation, and test datasets all have consistent feature dimensions, ensuring compatibility for model training and evaluation.

## 6. Baseline Model — sklearn Logistic Regression

A baseline logistic regression model is trained using the encoded feature set.
Model performance is evaluated on the validation dataset using AUC, AUCPR, log-loss, and macro F1.

An optimal probability threshold is selected based on the macro F1 score.

### 6.1 Evaluation Utilities

In [75]:
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, f1_score, confusion_matrix
import numpy as np

def find_best_f1_threshold(y_true, prob1, average="macro"):
    
    thresholds = np.linspace(0.01, 0.99, 99)
    f1_scores = []
    
    for t in thresholds:
        preds = (prob1 >= t).astype(int)
        f1_scores.append(f1_score(y_true, preds, average=average))
        
    best_idx = np.argmax(f1_scores)
    
    return thresholds[best_idx], f1_scores[best_idx]


def evaluate_model(y_true, prob1, threshold):
    
    auc = roc_auc_score(y_true, prob1)
    aucpr = average_precision_score(y_true, prob1)
    
    ll = log_loss(y_true, np.column_stack([1-prob1, prob1]))
    
    preds = (prob1 >= threshold).astype(int)
    
    f1 = f1_score(y_true, preds, average="macro")
    
    cm = confusion_matrix(y_true, preds)
    
    return auc, aucpr, ll, f1, cm

### 6.2 Train Baseline Logistic Regression

In [76]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
from sklearn.linear_model import LogisticRegression

lr_baseline = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=5000,
    random_state=42
)

lr_baseline.fit(X_train_enc, y_train)

val_prob1 = lr_baseline.predict_proba(X_val_enc)[:, 1]

baseline_threshold, best_f1 = find_best_f1_threshold(y_val, val_prob1, average="macro")
auc, aucpr, ll, f1_macro, cm = evaluate_model(y_val, val_prob1, baseline_threshold)

print("Baseline Logistic Regression (Validation)")
print("Baseline threshold:", round(baseline_threshold,4))
print("Macro F1:", round(f1_macro,4))
print("AUC:", round(auc,4))
print("AUCPR:", round(aucpr,4))
print("Log-loss:", round(ll,4))
print("Confusion Matrix:")
print(cm)

Baseline Logistic Regression (Validation)
Baseline threshold: 0.23
Macro F1: 0.5903
AUC: 0.6793
AUCPR: 0.2813
Log-loss: 0.438
Confusion Matrix:
[[104603  28870]
 [ 15936  12440]]


### Baseline Results (Validation)
A baseline sklearn Logistic Regression model was trained using the encoded feature set and evaluated on the validation dataset.

The probability threshold for class prediction was selected by maximizing **macro F1** on the validation set.

Baseline performance (validation):

- Best threshold: **0.23**
- Macro F1: **0.5903**
- AUC: **0.6793**
- AUCPR: **0.2813**
- Log-loss: **0.4380**
- Confusion matrix: **[[104603, 28870], [15936, 12440]]**

## 7. Model Tuning — sklearn Logistic Regression

Tune logistic regression over a grid of hyperparameters (≥ 50 configurations).  
Model selection is based on AUCPR evaluated on the validation dataset.  
After selecting the best configuration, compute the optimal classification threshold by maximizing macro F1 on the validation set.

In [77]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, log_loss
import pandas as pd
import numpy as np

# Hyperparameter grid (>= 50 combos)
Cs = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30]   # 10
class_weights = [None, "balanced"]                        # 2
tols = [1e-4, 3e-4, 1e-3]                                 # 3


results = []

for C in Cs:
    for cw in class_weights:
        for tol in tols:
            lr = LogisticRegression(
                C=C,
                solver="liblinear",
                max_iter=4000,
                tol=tol,
                class_weight=cw,
                random_state=42
            )

            lr.fit(X_train_enc, y_train)

            val_prob1 = lr.predict_proba(X_val_enc)[:, 1]

            results.append({
                "C": C,
                "class_weight": cw,
                "tol": tol,
                "solver": "liblinear",
                "val_aucpr": average_precision_score(y_val, val_prob1),
                "val_auc": roc_auc_score(y_val, val_prob1),
                "val_logloss": log_loss(y_val, np.column_stack([1 - val_prob1, val_prob1]))
            })

tune_df = pd.DataFrame(results).sort_values("val_aucpr", ascending=False)

print("Total hyperparameter combos tried:", tune_df.shape[0])
tune_df.head(10)

Total hyperparameter combos tried: 60


,C,class_weight,tol,solver,val_aucpr,val_auc,val_logloss
21,0.030,balanced,0.0001,liblinear,0.244542,0.652046,0.668796
33,0.300,balanced,0.0001,liblinear,0.244391,0.651910,0.668879
45,3.000,balanced,0.0001,liblinear,0.244153,0.651702,0.669001
9,0.003,balanced,0.0001,liblinear,0.244150,0.651698,0.669004
3,0.001,balanced,0.0001,liblinear,0.243891,0.651481,0.669134
39,1.000,balanced,0.0001,liblinear,0.243737,0.651338,0.669215
57,30.000,balanced,0.0001,liblinear,0.243736,0.651345,0.669212
27,0.100,balanced,0.0001,liblinear,0.243674,0.651289,0.669246
58,30.000,balanced,0.0003,liblinear,0.243632,0.651280,0.669275
51,10.000,balanced,0.0001,liblinear,0.243628,0.651248,0.669275


In [78]:
#Fit best tuned LR on full training set + choose threshold on validation

best = tune_df.iloc[0].to_dict()
print("Best config:", best)

best_lr = LogisticRegression(
    C=best["C"],
    solver="liblinear",
    max_iter=4000,
    tol=best["tol"],
    class_weight=best["class_weight"],
    random_state=42
)

best_lr.fit(X_train_enc, y_train)

val_prob1 = best_lr.predict_proba(X_val_enc)[:, 1]

tuned_threshold, best_f1 = find_best_f1_threshold(y_val, val_prob1, average="macro")
auc, aucpr, ll, f1m, cm = evaluate_model(y_val, val_prob1, tuned_threshold)

print("\nTuned Logistic Regression (Validation)")
print("Tuned threshold:", round(tuned_threshold,4))
print("Macro F1:", round(f1m,4))
print("AUC:", round(auc,4))
print("AUCPR:", round(aucpr,4))
print("Log-loss:", round(ll,4))
print("Confusion matrix:\n", cm)

Best config: {'C': 0.03, 'class_weight': 'balanced', 'tol': 0.0001, 'solver': 'liblinear', 'val_aucpr': 0.24454233818444787, 'val_auc': 0.6520460653031217, 'val_logloss': 0.6687963622971914}

Tuned Logistic Regression (Validation)
Tuned threshold: 0.57
Macro F1: 0.5605
AUC: 0.652
AUCPR: 0.2445
Log-loss: 0.6688
Confusion matrix:
 [[94799 38674]
 [14621 13755]]


### Logistic Regression Hyperparameter Tuning

A grid search was performed over 60 hyperparameter combinations using the validation set for model selection.  
The search varied regularization strength (C), tolerance (tol), and class_weight using the liblinear solver.

Model selection metric: **AUCPR**.

Best configuration:

- C = 0.03  
- solver = liblinear  
- class_weight = balanced  
- tol = 0.0001  

Validation performance of tuned model:

- Best threshold: 0.57  
- Macro F1: 0.5605  
- AUC: 0.6520  
- AUCPR: 0.2445  
- Log-loss: 0.6688  

Although hyperparameter tuning explored multiple configurations, the tuned model did not outperform the baseline logistic regression model. Therefore, the baseline configuration remains a strong reference point.

## 8. Final sklearn Model Selection and Test Evaluation

The baseline logistic regression model was selected as the final sklearn model because it achieved better validation performance than the tuned alternatives.
This section reports final performance on the held-out test set.

In [79]:
# Step 8: Evaluate FINAL sklearn model on TEST set 

# Refit baseline model on Train + Validation 
from sklearn.linear_model import LogisticRegression
import pandas as pd

X_trainval = pd.concat([X_train_enc, X_val_enc], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

final_lr = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=5000,
    random_state=42
)

final_lr.fit(X_trainval, y_trainval)

test_prob1 = final_lr.predict_proba(X_test_enc)[:, 1]

threshold_to_use = baseline_threshold  # from Step 6.2 baseline cell

auc, aucpr, ll, f1m, cm = evaluate_model(y_test, test_prob1, threshold_to_use)

print("FINAL sklearn Logistic Regression (TEST)")
print("Threshold used (from validation):", round(threshold_to_use,4))
print("Macro F1:", round(f1m,4))
print("AUC:", round(auc,4))
print("AUCPR:", round(aucpr,4))
print("Log-loss:", round(ll,4))
print("Confusion matrix:\n", cm)

FINAL sklearn Logistic Regression (TEST)
Threshold used (from validation): 0.23
Macro F1: 0.5914
AUC: 0.6786
AUCPR: 0.2787
Log-loss: 0.439
Confusion matrix:
 [[102252  31222]
 [ 14980  13396]]


### Final evaluation on Test Dataset (sklearn Logistic Regression)
The final sklearn Logistic Regression model is trained using the combined **training and validation datasets** to maximize the amount of data available for learning.

The probability threshold used for class assignment is the **baseline threshold selected on the validation set (0.23).** The test dataset is strictly held out and used **only once for final evaluation**, ensuring no data leakage during model development.

The model predictions on the test dataset are evaluated using the required metrics:
* AUC
* AUCPR
* Log-loss
* Confusion Matrix at the optimized threshold

The results provide an unbiased estimate of how the model would perform on unseen data in a real-world deployment scenario.


## 9. H2O-3 GLM (Binomial) — Training and Tuning
This section trains and tunes an H2O-3 GLM (binomial logistic regression) as the second required linear-model family.
Hyperparameters are selected using the validation set and AUCPR as the primary selection metric.
The final classification threshold is chosen by maximizing macro-F1 on the validation set and then applied unchanged to the test set.


### 9.1 Start H2O

In [80]:
#Start H2O
import h2o
from h2o.estimators.glm import H2OGeneralizedLinearEstimator

h2o.init(max_mem_size="6G")   
h2o.no_progress()             

Checking whether there is an H2O instance running at http://localhost:54321. connected.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,1 day 8 hours 57 mins
H2O_cluster_timezone:,America/Chicago
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,3 months and 23 days
H2O_cluster_name:,H2O_from_python_himavarshakomanduri_9o7ifs
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,4.771 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


### 9.2 Create a tuning subset (compute efficiency)

The full training set is large, so hyperparameter search is performed on a stratified sample of the training data to reduce runtime.
After selecting the best configuration, the best-performing model is used for validation/test evaluation.

In [81]:
#Create a tuning subset
from sklearn.model_selection import train_test_split

X_tune_pd, _, y_tune_pd, _ = train_test_split(
    X_train_enc, y_train,
    train_size=120000,
    stratify=y_train,
    random_state=42
)

print("Tune sample:", X_tune_pd.shape)
print("Tune balance:\n", y_tune_pd.value_counts(normalize=True))

Tune sample: (120000, 52)
Tune balance:
 MIS_Status
0    0.824675
1    0.175325
Name: proportion, dtype: float64


### 9.3 Build H2OFrames with consistent numeric types

One-hot encoded features are numeric (0/1), but H2O can sometimes infer different column types across frames.
To prevent schema/type mismatch errors, features are coerced to numeric in pandas before conversion to H2OFrames.

In [82]:
import pandas as pd
import h2o

X_tune_pd_num = X_tune_pd.apply(pd.to_numeric, errors="coerce")
X_val_pd_num  = X_val_enc.apply(pd.to_numeric, errors="coerce")

train_tune_h2o = h2o.H2OFrame(pd.concat([X_tune_pd_num, y_tune_pd.rename("MIS_Status")], axis=1))
val_h2o        = h2o.H2OFrame(pd.concat([X_val_pd_num,  y_val.rename("MIS_Status")], axis=1))

train_tune_h2o["MIS_Status"] = train_tune_h2o["MIS_Status"].asfactor()
val_h2o["MIS_Status"]        = val_h2o["MIS_Status"].asfactor()

x_cols = [c for c in train_tune_h2o.columns if c != "MIS_Status"]
y_col = "MIS_Status"

print("Tune H2O shape:", train_tune_h2o.dim)
print("Val H2O shape:", val_h2o.dim)
print("Num features:", len(x_cols))

Tune H2O shape: [120000, 53]
Val H2O shape: [161849, 53]
Num features: 52


### 9.4 Remove constant features in tuning sample

Some engineered / one-hot columns may be constant in the tuning sample (no variance). These are removed from both tune and validation frames to keep schemas consistent.

In [83]:
# Identify constant columns in the tuning sample 
const_cols = [c for c in X_tune_pd_num.columns if X_tune_pd_num[c].nunique() <= 1]

print("Constant columns removed:", const_cols)
print("Count:", len(const_cols))

X_tune_pd_num = X_tune_pd_num.drop(columns=const_cols)
X_val_pd_num  = X_val_pd_num.drop(columns=const_cols)

# Rebuild H2OFrames after dropping constants
train_tune_h2o = h2o.H2OFrame(pd.concat([X_tune_pd_num, y_tune_pd.rename("MIS_Status")], axis=1))
val_h2o        = h2o.H2OFrame(pd.concat([X_val_pd_num,  y_val.rename("MIS_Status")], axis=1))

train_tune_h2o["MIS_Status"] = train_tune_h2o["MIS_Status"].asfactor()
val_h2o["MIS_Status"]        = val_h2o["MIS_Status"].asfactor()

x_cols = [c for c in train_tune_h2o.columns if c != "MIS_Status"]
y_col = "MIS_Status"

print("Tune H2O shape:", train_tune_h2o.dim)
print("Val H2O shape:", val_h2o.dim)
print("Num features:", len(x_cols))

Constant columns removed: ['BalanceGross', 'balance_to_disbursement', 'RevLineCr_-', 'RevLineCr_.', 'RevLineCr_3', 'RevLineCr_4', 'RevLineCr_7', 'RevLineCr_A', 'RevLineCr_C', 'LowDoc_1']
Count: 10
Tune H2O shape: [120000, 43]
Val H2O shape: [161849, 43]
Num features: 42


### 9.5 Fix H2O column type mismatch

H2O requires identical column types across training and validation frames.
A one-hot column (RevLineCr_0) was inferred as categorical in one frame and numeric in another, which causes a schema mismatch during model training.
The column is explicitly cast to numeric in both frames to ensure consistent data types.

In [107]:
# Force the mismatched column to numeric in BOTH frames
train_tune_h2o["RevLineCr_0"] = train_tune_h2o["RevLineCr_0"].asnumeric()
val_h2o["RevLineCr_0"]        = val_h2o["RevLineCr_0"].asnumeric()

print("train type:", train_tune_h2o.types["RevLineCr_0"])
print("val type:", val_h2o.types["RevLineCr_0"])

train type: real
val type: real


### 9.6 Hyperparameter tuning for H2O GLM

A grid search is used to tune the H2O GLM model.  
Four parameters are explored:

- **alpha** – controls elastic-net mixing between L1 and L2 regularization  
- **lambda** – regularization strength  
- **standardize** – whether features are standardized inside H2O  
- **missing_values_handling** – strategy for handling missing values

The grid contains **60 total combinations**, satisfying the project requirement of evaluating at least 50 model configurations.

Each configuration is trained on the tuning sample and evaluated on the validation set.  
Models are ranked primarily by **validation AUCPR**, with AUC and log-loss also recorded.

In [85]:
import itertools
import pandas as pd
from h2o.estimators.glm import H2OGeneralizedLinearEstimator

alpha_list = [0.0, 0.5, 1.0]                      
lambda_list = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2]      
standardize_list = [True, False]                  
mvh_list = ["MeanImputation", "Skip"]             

grid = list(itertools.product(alpha_list, lambda_list, standardize_list, mvh_list))
print("Total combos:", len(grid))  

results = []

for alpha, lam, std, mvh in grid:
    glm = H2OGeneralizedLinearEstimator(
        family="binomial",
        alpha=alpha,
        lambda_=lam,
        standardize=std,
        missing_values_handling=mvh,
        max_iterations=200,
        seed=42
    )
    glm.train(x=x_cols, y=y_col, training_frame=train_tune_h2o, validation_frame=val_h2o)

    perf = glm.model_performance(val_h2o)
    results.append({
        "alpha": alpha,
        "lambda": lam,
        "standardize": std,
        "missing_values_handling": mvh,
        "val_auc": perf.auc(),
        "val_aucpr": perf.aucpr(),
        "val_logloss": perf.logloss(),
        "model_id": glm.model_id
    })

h2o_tune_df = pd.DataFrame(results).sort_values("val_aucpr", ascending=False)

print("Total models trained:", h2o_tune_df.shape[0])
h2o_tune_df.head(10)

Total combos: 60
Total models trained: 60


,alpha,lambda,standardize,missing_values_handling,val_auc,val_aucpr,val_logloss,model_id
34,0.5,0.0010,False,MeanImputation,0.771479,0.445119,0.391186,GLM_model_python_1773860887253_857
35,0.5,0.0010,False,Skip,0.771479,0.445119,0.391186,GLM_model_python_1773860887253_861
14,0.0,0.0010,False,MeanImputation,0.770177,0.444036,0.391881,GLM_model_python_1773860887253_777
15,0.0,0.0010,False,Skip,0.770177,0.444036,0.391881,GLM_model_python_1773860887253_781
54,1.0,0.0010,False,MeanImputation,0.770929,0.442978,0.391302,GLM_model_python_1773860887253_937
55,1.0,0.0010,False,Skip,0.770929,0.442978,0.391302,GLM_model_python_1773860887253_941
10,0.0,0.0001,False,MeanImputation,0.770458,0.442061,0.391805,GLM_model_python_1773860887253_761
11,0.0,0.0001,False,Skip,0.770458,0.442061,0.391805,GLM_model_python_1773860887253_765
31,0.5,0.0001,False,Skip,0.769766,0.440626,0.392364,GLM_model_python_1773860887253_845
30,0.5,0.0001,False,MeanImputation,0.769766,0.440626,0.392364,GLM_model_python_1773860887253_841


### 9.7 Select the best tuned H2O GLM model (by validation AUCPR)

The best-performing configuration from the tuning grid is selected using validation **AUCPR**.
Predicted probabilities (p1) are generated on the validation set for threshold selection in the next step.

In [86]:
# Get best model id
best_model_id = h2o_tune_df.iloc[0]["model_id"]

print("Best model:", best_model_id)

best_glm = h2o.get_model(best_model_id)

# Validation predictions
val_pred_h2o = best_glm.predict(val_h2o)

val_pred = val_pred_h2o.as_data_frame()["p1"].values

Best model: GLM_model_python_1773860887253_857


/Users/himavarshakomanduri/Desktop/Spring 2026/BUAN 6341 - Applied Machine Learning/.venv/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


### 9.8 Evaluation utilities and threshold selection

To maintain consistent evaluation across models, helper functions are defined for:

- selecting the optimal classification threshold based on **macro-F1**
- computing evaluation metrics (AUC, AUCPR, log-loss, macro-F1, confusion matrix)

These functions are reused for both validation and test evaluation.

In [87]:
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, f1_score, confusion_matrix
import numpy as np

def find_best_f1_threshold(y_true, prob, average="macro"):
    thresholds = np.linspace(0.01, 0.99, 99)
    best_t = 0.5
    best_f1 = -1

    for t in thresholds:
        preds = (prob >= t).astype(int)
        f1 = f1_score(y_true, preds, average=average)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1


def evaluate_model(y_true, prob, threshold):
    preds = (prob >= threshold).astype(int)

    auc = roc_auc_score(y_true, prob)
    aucpr = average_precision_score(y_true, prob)
    ll = log_loss(y_true, np.column_stack([1 - prob, prob]))
    f1_macro = f1_score(y_true, preds, average="macro")
    cm = confusion_matrix(y_true, preds)

    return auc, aucpr, ll, f1_macro, cm

### 9.9 Select optimal classification threshold (validation set)

The threshold is chosen using the validation set by maximizing **macro-F1**.
This threshold is then fixed and applied to the test set to avoid test-set leakage.

In [88]:
h2o_best_threshold, h2o_best_f1 = find_best_f1_threshold(y_val.values, val_pred, average="macro")

auc, aucpr, ll, f1_macro, cm = evaluate_model(y_val.values, val_pred, h2o_best_threshold)

print("H2O GLM (Validation)")
print("Best threshold:", round(h2o_best_threshold,4))
print("Macro F1:", round(f1_macro,4))
print("AUC:", round(auc,4))
print("AUCPR:", round(aucpr,4))
print("Log-loss:", round(ll,4))
print("Confusion matrix:")
print(cm)

H2O GLM (Validation)
Best threshold: 0.29
Macro F1: 0.6677
AUC: 0.7715
AUCPR: 0.4452
Log-loss: 0.3912
Confusion matrix:
[[117737  15736]
 [ 15476  12900]]


### 9.10 Final evaluation on the held-out test set

The tuned H2O GLM is evaluated once on the test set.  
The classification threshold selected on the validation set is applied unchanged to avoid test-set leakage.

In [89]:
#Evaluate model on Test
# Test predictions
test_h2o = h2o.H2OFrame(pd.concat([X_test_enc.apply(pd.to_numeric, errors="coerce"),
                                   y_test.rename("MIS_Status")], axis=1))

test_h2o["MIS_Status"] = test_h2o["MIS_Status"].asfactor()

test_pred_h2o = best_glm.predict(test_h2o)

test_prob = test_pred_h2o.as_data_frame()["p1"].values

auc, aucpr, ll, f1_macro, cm = evaluate_model(y_test.values, test_prob, h2o_best_threshold)

print("FINAL H2O GLM (TEST)")
print("Threshold used (from validation):", h2o_best_threshold)
print("Macro F1:", round(f1_macro,4))
print("AUC:", round(auc,4))
print("AUCPR:", round(aucpr,4))
print("Log-loss:", round(ll,4))
print("Confusion matrix:\n", cm)

/Users/himavarshakomanduri/Desktop/Spring 2026/BUAN 6341 - Applied Machine Learning/.venv/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


FINAL H2O GLM (TEST)
Threshold used (from validation): 0.29000000000000004
Macro F1: 0.6664
AUC: 0.7713
AUCPR: 0.4437
Log-loss: 0.3914
Confusion matrix:
 [[117693  15781]
 [ 15548  12828]]


## 10. Project Summary, Final Model Choice, and Next Steps

Two required linear-model families were trained and tuned: (1) sklearn Logistic Regression and (2) H2O-3 GLM (binomial). 
Categorical features were encoded using one-hot (low-cardinality) and supervised encoders (Target Encoding / WOE) fit on the training set only to avoid leakage. 
At least 10 engineered features were added prior to model training.

For sklearn Logistic Regression, a hyperparameter search (≥50 configurations) was performed and models were compared using validation AUCPR (and log-loss recorded). 
The best classification threshold was selected on the validation set by maximizing macro-F1, then applied unchanged to the test set.

For H2O-3 GLM, a 60-combination grid search over regularization and preprocessing settings (alpha, lambda, standardize, missing value handling) was run and ranked by validation AUCPR.
The final H2O model achieved strong and consistent performance from validation to test.

**Final model selected for deployment and scoring:** H2O-3 GLM  
**Threshold policy:** the classification threshold was selected on the validation set by maximizing macro-F1 (selected threshold = 0.29) and then applied unchanged to the test and holdout datasets.

**Final test performance (H2O GLM):** AUC = 0.7713, AUCPR = 0.4437, Log-loss = 0.3914, Macro-F1 = 0.6664.  
This outperformed the sklearn Logistic Regression model on ranking metrics (AUC/AUCPR), while retaining interpretability expected from linear models.

**Risks / considerations:** performance may shift if future data has different distributions (e.g., geography, bank mix, or loan amounts). Threshold choice also affects the tradeoff between false positives and false negatives, which should be aligned with business cost.

**Next steps:**     
 (1) calibrate probabilities and evaluate threshold under cost-sensitive objectives;        
 (2) perform coefficient review / stability checks and finalize a reproducible scoring pipeline for the unlabeled holdout dataset.

## 11. Save Model Artifacts for Scoring

To support reproducible scoring on unseen datasets, the trained models and preprocessing artifacts are saved after model evaluation.

Artifacts saved include:

- Trained sklearn Logistic Regression model
- Trained H2O-3 GLM model (final model selected for deployment)
- Fitted Target Encoder and WOE Encoder objects
- Final feature column list used during model training
- Optimal probability threshold determined on the validation set

Saving these artifacts ensures that the same preprocessing, feature engineering, encoding, and model configuration used during training can be applied consistently during inference.

All artifacts are stored in the `artifacts/` directory and will be loaded by the scoring function implemented in `scoring_template.py`.

In [90]:
import os
import pickle
import h2o

os.makedirs("artifacts", exist_ok=True)

# ---------- sklearn artifacts ----------
sklearn_artifacts = {
    "model": lr_baseline,
    "threshold": 0.23,
    "feature_columns": list(X_train_enc.columns),
    "target_encoder": target_encoder,
    "woe_encoder": woe_encoder
}

with open("artifacts/sklearn_artifacts.pkl", "wb") as f:
    pickle.dump(sklearn_artifacts, f)

print("Saved sklearn artifacts")


# ---------- H2O artifacts ----------
h2o_model_path = h2o.save_model(best_glm, path="artifacts", force=True)

h2o_artifacts = {
    "model_path": h2o_model_path,
    "threshold": 0.29,
    "feature_columns": list(X_train_enc.columns),
    "target_encoder": target_encoder,
    "woe_encoder": woe_encoder
}

with open("artifacts/h2o_artifacts.pkl", "wb") as f:
    pickle.dump(h2o_artifacts, f)

print("Saved H2O artifacts")
print("H2O model path:", h2o_model_path)

Saved sklearn artifacts
Saved H2O artifacts
H2O model path: /Users/himavarshakomanduri/Desktop/Spring 2026/BUAN 6341 - Applied Machine Learning/project-1-khvarsha16-web/artifacts/GLM_model_python_1773860887253_857


## 12. Scoring Function Validation

In this step, we validate the `project_1_scoring` function using the holdout dataset provided for Project 1.

The scoring function performs the following steps:

- Loads saved model artifacts (model, encoders, feature columns, and threshold)
- Applies the same preprocessing and feature engineering used during training
- Applies categorical encoding using the previously fitted encoders
- Generates predictions for each record in the holdout dataset
- Returns results in the required format:

**index, label, probability_0, probability_1**

This validation ensures that the scoring pipeline correctly reproduces the training pipeline and can generate predictions for unseen data.

In [91]:
import pandas as pd
from scoring_template import project_1_scoring

holdout = pd.read_csv("./data/SBA_loans_project_1_holdout_students_valid_no_labels.csv.zip")
preds = project_1_scoring(holdout, model_type="sklearn")

print(preds.head())
print(preds.shape)
print(preds.isna().sum())

   index  label  probability_0  probability_1
0      0      0       0.870228       0.129772
1      1      1       0.748157       0.251843
2      2      0       0.803798       0.196202
3      3      0       0.832380       0.167620
4      4      0       0.878632       0.121368
(89917, 4)
index            0
label            0
probability_0    0
probability_1    0
dtype: int64


### Scoring Validation Results

The scoring function successfully generated predictions for the entire holdout dataset.

Validation checks confirm that:

- The output shape matches the input dataset (89917 rows).
- All required columns are present.
- No missing values were produced in the prediction output.
- The predicted probabilities sum correctly for each observation.

These checks confirm that the scoring pipeline correctly reproduces the preprocessing, feature engineering, encoding, and prediction steps used during model training and is ready for deployment on unseen data.

In [94]:
submission = preds[["index", "probability_1"]].copy()

# rename column
submission.columns = ["id", "probability_1"]

# reset index to avoid weird alignment issues
submission = submission.reset_index(drop=True)

# save clean file
submission.to_csv("submission.csv", index=False)

print(submission.shape)
print(submission.isna().sum())

(89917, 2)
id               0
probability_1    0
dtype: int64


In [95]:
submission = preds[["index", "probability_1"]].copy()

# FIX → column name must be uppercase ID
submission.columns = ["ID", "probability_1"]

submission.to_csv("submission.csv", index=False)

print(submission.head())
print(submission.shape)

   ID  probability_1
0   0       0.129772
1   1       0.251843
2   2       0.196202
3   3       0.167620
4   4       0.121368
(89917, 2)


In [96]:
print(submission.shape)
print(submission.isna().sum())

(89917, 2)
ID               0
probability_1    0
dtype: int64


In [105]:
import pandas as pd
import numpy as np
import h2o

# ── 1. Reload holdout ──────────────────────────────────────────────────────────
holdout = pd.read_csv("./data/SBA_loans_project_1_holdout_students_valid_no_labels.csv.zip").copy()

# ── 2. Fill missing values (same as training pipeline) ─────────────────────────
cat_fill_unknown = ["RevLineCr", "LowDoc", "BankState", "Bank", "City", "State"]
for col in cat_fill_unknown:
    if col in holdout.columns:
        holdout[col] = holdout[col].fillna("Unknown")
holdout["NewExist"] = holdout["NewExist"].fillna(holdout["NewExist"].mode()[0])

# ── 3. Coerce numerics ─────────────────────────────────────────────────────────
num_cols = ["SBA_Appv","GrAppv","DisbursementGross","BalanceGross",
            "CreateJob","RetainedJob","NoEmp","UrbanRural","FranchiseCode"]
for c in num_cols:
    if c in holdout.columns:
        holdout[c] = pd.to_numeric(holdout[c], errors="coerce")

# ── 4. Feature engineering (identical to training) ────────────────────────────
holdout["loan_to_sba_ratio"]          = holdout["SBA_Appv"] / (holdout["GrAppv"] + 1)
holdout["disbursement_to_approval"]   = holdout["DisbursementGross"] / (holdout["GrAppv"] + 1)
holdout["balance_to_disbursement"]    = holdout["BalanceGross"] / (holdout["DisbursementGross"] + 1)
holdout["jobs_created_ratio"]         = holdout["CreateJob"] / (holdout["NoEmp"] + 1)
holdout["jobs_retained_ratio"]        = holdout["RetainedJob"] / (holdout["NoEmp"] + 1)
holdout["total_jobs"]                 = holdout["CreateJob"] + holdout["RetainedJob"]
holdout["jobs_per_employee"]          = (holdout["CreateJob"] + holdout["RetainedJob"]) / (holdout["NoEmp"] + 1)
holdout["is_urban"]                   = (holdout["UrbanRural"] == 1).astype(int)
holdout["is_franchise"]               = (holdout["FranchiseCode"] > 0).astype(int)
holdout["sba_guarantee_diff"]         = holdout["GrAppv"] - holdout["SBA_Appv"]
holdout["log_approval"]               = np.log1p(holdout["GrAppv"])

# ── 5. One-hot encode (align to training columns BEFORE target encoding) ───────
holdout_oh = pd.get_dummies(holdout, columns=["RevLineCr", "LowDoc"], drop_first=True)
holdout_oh = holdout_oh.reindex(columns=X_train_oh.columns, fill_value=0)  # align to train

# ── 6. Target + WOE encoding ──────────────────────────────────────────────────
holdout_te  = target_encoder.transform(holdout_oh)
holdout_enc = woe_encoder.transform(holdout_te)

# Align to final training encoded columns
holdout_enc = holdout_enc.reindex(columns=X_train_enc.columns, fill_value=0)

# ── 7. Drop constant columns that were removed before H2O training ─────────────
drop_cols = [c for c in const_cols if c in holdout_enc.columns]
holdout_enc = holdout_enc.drop(columns=drop_cols)

print("Holdout encoded shape:", holdout_enc.shape)
print("Expected H2O columns:", len(x_cols))
print("Shapes match:", list(holdout_enc.columns) == list(x_cols))

# ── 8. Convert to H2OFrame ────────────────────────────────────────────────────
holdout_num = holdout_enc.apply(pd.to_numeric, errors="coerce")
holdout_h2o_final = h2o.H2OFrame(holdout_num)

# Fix the RevLineCr_0 type mismatch (same fix as during training)
if "RevLineCr_0" in holdout_h2o_final.columns:
    holdout_h2o_final["RevLineCr_0"] = holdout_h2o_final["RevLineCr_0"].asnumeric()

# ── 9. Predict ─────────────────────────────────────────────────────────────────
h2o_holdout_preds = best_glm.predict(holdout_h2o_final).as_data_frame()
print(h2o_holdout_preds.head())

# ── 10. Build submission CSV ──────────────────────────────────────────────────
holdout_ids = pd.read_csv("./data/SBA_loans_project_1_holdout_students_valid_no_labels.csv.zip")["index"]

submission_h2o = pd.DataFrame({
    "id": holdout_ids.values,
    "probability_1": h2o_holdout_preds["p1"].values
})

submission_h2o.to_csv("submission_h2o.csv", index=False)
print(submission_h2o.head())
print(submission_h2o.shape)
print(submission_h2o.isna().sum())

Holdout encoded shape: (89917, 42)
Expected H2O columns: 42
Shapes match: True


OSError: Job with key $03017f00000132d4ffffffff$_ad1f5d6ce003d754c539ac7130de4f35 failed with an exception: java.lang.IllegalArgumentException: Test/Validation dataset has a non-categorical column 'RevLineCr_`' which is categorical in the training data
stacktrace: 
java.lang.IllegalArgumentException: Test/Validation dataset has a non-categorical column 'RevLineCr_`' which is categorical in the training data
	at hex.Model.adaptTestForTrain(Model.java:1820)
	at hex.Model.adaptTestForTrain(Model.java:1643)
	at hex.Model.adaptTestForTrain(Model.java:1639)
	at hex.Model.adaptFrameForScore(Model.java:1981)
	at hex.Model.score(Model.java:2011)
	at water.api.ModelMetricsHandler$1.compute2(ModelMetricsHandler.java:555)
	at water.H2O$H2OCountedCompleter.compute(H2O.java:1704)
	at jsr166y.CountedCompleter.exec(CountedCompleter.java:468)
	at jsr166y.ForkJoinTask.doExec(ForkJoinTask.java:263)
	at jsr166y.ForkJoinPool$WorkQueue.runTask(ForkJoinPool.java:976)
	at jsr166y.ForkJoinPool.runWorker(ForkJo

In [106]:
# ── 8. Convert to H2OFrame — force ALL columns to numeric ─────────────────────
holdout_num = holdout_enc.apply(pd.to_numeric, errors="coerce").astype(float)

holdout_h2o_final = h2o.H2OFrame(holdout_num)

# Force every column to numeric in H2O (catches any H2O mis-inferences)
for col in holdout_h2o_final.columns:
    if holdout_h2o_final[col].isfactor()[0]:
        holdout_h2o_final[col] = holdout_h2o_final[col].asnumeric()

print("Any remaining factor columns?", 
      [c for c in holdout_h2o_final.columns if holdout_h2o_final[c].isfactor()[0]])

# ── 9. Predict ─────────────────────────────────────────────────────────────────
h2o_holdout_preds = best_glm.predict(holdout_h2o_final).as_data_frame()
print(h2o_holdout_preds.head())

Any remaining factor columns? []


OSError: Job with key $03017f00000132d4ffffffff$_ac4a69560a8cb161d3594671507d435f failed with an exception: java.lang.IllegalArgumentException: Test/Validation dataset has a non-categorical column 'RevLineCr_1' which is categorical in the training data
stacktrace: 
java.lang.IllegalArgumentException: Test/Validation dataset has a non-categorical column 'RevLineCr_1' which is categorical in the training data
	at hex.Model.adaptTestForTrain(Model.java:1820)
	at hex.Model.adaptTestForTrain(Model.java:1643)
	at hex.Model.adaptTestForTrain(Model.java:1639)
	at hex.Model.adaptFrameForScore(Model.java:1981)
	at hex.Model.score(Model.java:2011)
	at water.api.ModelMetricsHandler$1.compute2(ModelMetricsHandler.java:555)
	at water.H2O$H2OCountedCompleter.compute(H2O.java:1704)
	at jsr166y.CountedCompleter.exec(CountedCompleter.java:468)
	at jsr166y.ForkJoinTask.doExec(ForkJoinTask.java:263)
	at jsr166y.ForkJoinPool$WorkQueue.runTask(ForkJoinPool.java:976)
	at jsr166y.ForkJoinPool.runWorker(ForkJo

In [108]:
# ── 8. Convert to H2OFrame and match types to training frame exactly ───────────
holdout_num = holdout_enc.apply(pd.to_numeric, errors="coerce").astype(float)
holdout_h2o_final = h2o.H2OFrame(holdout_num)

# Match every column's type to what the training frame used
for col in x_cols:
    if col not in holdout_h2o_final.columns:
        continue
    train_type = train_tune_h2o.types[col]
    holdout_type = holdout_h2o_final.types[col]
    if train_type != holdout_type:
        print(f"Fixing {col}: holdout={holdout_type} → train={train_type}")
        if train_type == "enum":
            holdout_h2o_final[col] = holdout_h2o_final[col].asfactor()
        else:
            holdout_h2o_final[col] = holdout_h2o_final[col].asnumeric()

# Verify no mismatches remain
mismatches = [c for c in x_cols if train_tune_h2o.types[c] != holdout_h2o_final.types[c]]
print("Remaining mismatches:", mismatches)

# ── 9. Predict ─────────────────────────────────────────────────────────────────
h2o_holdout_preds = best_glm.predict(holdout_h2o_final).as_data_frame()
print(h2o_holdout_preds.head())

# ── 10. Build submission CSV ───────────────────────────────────────────────────
holdout_ids = pd.read_csv("./data/SBA_loans_project_1_holdout_students_valid_no_labels.csv.zip")["index"]

submission_h2o = pd.DataFrame({
    "id": holdout_ids.values,
    "probability_1": h2o_holdout_preds["p1"].values
})

submission_h2o.to_csv("submission_h2o.csv", index=False)
print(submission_h2o.head())
print(submission_h2o.shape)
print(submission_h2o.isna().sum())

Fixing RevLineCr_0: holdout=int → train=real
Fixing RevLineCr_1: holdout=int → train=enum
Fixing RevLineCr_2: holdout=int → train=enum
Fixing RevLineCr_N: holdout=int → train=enum
Fixing RevLineCr_R: holdout=int → train=enum
Fixing RevLineCr_T: holdout=int → train=enum
Fixing RevLineCr_Unknown: holdout=int → train=enum
Fixing RevLineCr_Y: holdout=int → train=enum
Fixing RevLineCr_`: holdout=int → train=enum
Fixing LowDoc_A: holdout=int → train=enum
Fixing LowDoc_C: holdout=int → train=enum
Fixing LowDoc_N: holdout=int → train=enum
Fixing LowDoc_R: holdout=int → train=enum
Fixing LowDoc_S: holdout=int → train=enum
Fixing LowDoc_Unknown: holdout=int → train=enum
Fixing LowDoc_Y: holdout=int → train=enum
Remaining mismatches: []
   predict        p0        p1
0        0  0.897458  0.102542
1        0  0.940398  0.059602
2        0  0.809097  0.190903
3        0  0.757317  0.242683
4        0  0.900711  0.099289


/Users/himavarshakomanduri/Desktop/Spring 2026/BUAN 6341 - Applied Machine Learning/.venv/lib/python3.12/site-packages/h2o/job.py:81: UserWarning: Test/Validation dataset column 'RevLineCr_1' has levels not trained on: ["0", "1"]
  warnings.warn(w)
/Users/himavarshakomanduri/Desktop/Spring 2026/BUAN 6341 - Applied Machine Learning/.venv/lib/python3.12/site-packages/h2o/job.py:81: UserWarning: Test/Validation dataset column 'RevLineCr_2' has levels not trained on: ["0", "1"]
  warnings.warn(w)
/Users/himavarshakomanduri/Desktop/Spring 2026/BUAN 6341 - Applied Machine Learning/.venv/lib/python3.12/site-packages/h2o/job.py:81: UserWarning: Test/Validation dataset column 'RevLineCr_N' has levels not trained on: ["0", "1"]
  warnings.warn(w)
/Users/himavarshakomanduri/Desktop/Spring 2026/BUAN 6341 - Applied Machine Learning/.venv/lib/python3.12/site-packages/h2o/job.py:81: UserWarning: Test/Validation dataset column 'RevLineCr_R' has levels not trained on: ["0", "1"]
  warnings.warn(w)
/Use

   id  probability_1
0   0       0.102542
1   1       0.059602
2   2       0.190903
3   3       0.242683
4   4       0.099289
(89917, 2)
id               0
probability_1    0
dtype: int64


In [109]:
submission_h2o = pd.DataFrame({
    "ID": holdout_ids.values,
    "probability_1": h2o_holdout_preds["p1"].values
})

submission_h2o.to_csv("submission_.csv", index=False)
print(submission_h2o.head())

   ID  probability_1
0   0       0.102542
1   1       0.059602
2   2       0.190903
3   3       0.242683
4   4       0.099289


In [111]:
# ── Retrain on ALL training data ───────────────────────────────────────────────
X_full_num = X_train_enc.apply(pd.to_numeric, errors="coerce")

full_h2o = h2o.H2OFrame(pd.concat([X_full_num, y_train.rename("MIS_Status")], axis=1))
full_h2o["MIS_Status"] = full_h2o["MIS_Status"].asfactor()

# Fix type mismatches same as before
for col in full_h2o.columns:
    if col == "MIS_Status":
        continue
    if full_h2o[col].isfactor()[0]:
        full_h2o[col] = full_h2o[col].asnumeric()

# Use best config from tuning
best_config = h2o_tune_df.iloc[0]
final_glm_full = H2OGeneralizedLinearEstimator(
    family="binomial",
    alpha=float(best_config["alpha"]),
    lambda_=float(best_config["lambda"]),
    standardize=bool(best_config["standardize"]),
    missing_values_handling=str(best_config["missing_values_handling"]),
    max_iterations=200,
    seed=42
)

final_glm_full.train(x=x_cols, y="MIS_Status", training_frame=full_h2o)
print("Done training on full data")

# Validate it still makes sense on val set
val_preds_full = final_glm_full.predict(val_h2o).as_data_frame()["p1"].values
auc, aucpr, ll, f1m, cm = evaluate_model(y_val.values, val_preds_full, h2o_best_threshold)
print(f"Val AUCPR (full retrain): {aucpr:.4f}  AUC: {auc:.4f}")

Done training on full data


OSError: Job with key $03017f00000132d4ffffffff$_8c451c82124443012c86c9379df54fb4 failed with an exception: java.lang.IllegalArgumentException: Test/Validation dataset has categorical column 'RevLineCr_1' which is real-valued in the training data
stacktrace: 
java.lang.IllegalArgumentException: Test/Validation dataset has categorical column 'RevLineCr_1' which is real-valued in the training data
	at hex.Model.adaptTestForTrain(Model.java:1833)
	at hex.Model.adaptTestForTrain(Model.java:1643)
	at hex.Model.adaptTestForTrain(Model.java:1639)
	at hex.Model.adaptFrameForScore(Model.java:1981)
	at hex.Model.score(Model.java:2011)
	at water.api.ModelMetricsHandler$1.compute2(ModelMetricsHandler.java:555)
	at water.H2O$H2OCountedCompleter.compute(H2O.java:1704)
	at jsr166y.CountedCompleter.exec(CountedCompleter.java:468)
	at jsr166y.ForkJoinTask.doExec(ForkJoinTask.java:263)
	at jsr166y.ForkJoinPool$WorkQueue.runTask(ForkJoinPool.java:976)
	at jsr166y.ForkJoinPool.runWorker(ForkJoinPool.java:

In [112]:
# Fix val_h2o types to match full_h2o
for col in x_cols:
    if col not in val_h2o.columns:
        continue
    train_type = full_h2o.types[col]
    val_type = val_h2o.types[col]
    if train_type != val_type:
        print(f"Fixing val {col}: {val_type} → {train_type}")
        if train_type == "enum":
            val_h2o[col] = val_h2o[col].asfactor()
        else:
            val_h2o[col] = val_h2o[col].asnumeric()

# Now predict
val_preds_full = final_glm_full.predict(val_h2o).as_data_frame()["p1"].values
auc, aucpr, ll, f1m, cm = evaluate_model(y_val.values, val_preds_full, h2o_best_threshold)
print(f"Val AUCPR (full retrain): {aucpr:.4f}  AUC: {auc:.4f}")

Fixing val RevLineCr_1: enum → real
Fixing val RevLineCr_2: enum → real
Fixing val RevLineCr_N: enum → real
Fixing val RevLineCr_R: enum → real
Fixing val RevLineCr_T: enum → real
Fixing val RevLineCr_Unknown: enum → real
Fixing val RevLineCr_Y: enum → real
Fixing val RevLineCr_`: enum → real
Fixing val LowDoc_A: enum → real
Fixing val LowDoc_C: enum → real
Fixing val LowDoc_N: enum → real
Fixing val LowDoc_R: enum → real
Fixing val LowDoc_S: enum → real
Fixing val LowDoc_Unknown: enum → real
Fixing val LowDoc_Y: enum → real


/Users/himavarshakomanduri/Desktop/Spring 2026/BUAN 6341 - Applied Machine Learning/.venv/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


Val AUCPR (full retrain): 0.4457  AUC: 0.7712


In [113]:
# Fix holdout types to match full_h2o
for col in x_cols:
    if col not in holdout_h2o_final.columns:
        continue
    train_type = full_h2o.types[col]
    holdout_type = holdout_h2o_final.types[col]
    if train_type != holdout_type:
        if train_type == "enum":
            holdout_h2o_final[col] = holdout_h2o_final[col].asfactor()
        else:
            holdout_h2o_final[col] = holdout_h2o_final[col].asnumeric()

# Predict with full retrained model
h2o_holdout_preds_full = final_glm_full.predict(holdout_h2o_final).as_data_frame()

# Build submission
holdout_ids = pd.read_csv("./data/SBA_loans_project_1_holdout_students_valid_no_labels.csv.zip")["index"]

submission_full = pd.DataFrame({
    "ID": holdout_ids.values,
    "probability_1": h2o_holdout_preds_full["p1"].values
})

submission_full.to_csv("submission_full_retrain.csv", index=False)
print(submission_full.head())
print(submission_full.shape)
print(submission_full.isna().sum())

/Users/himavarshakomanduri/Desktop/Spring 2026/BUAN 6341 - Applied Machine Learning/.venv/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


   ID  probability_1
0   0       0.103204
1   1       0.062142
2   2       0.185788
3   3       0.223318
4   4       0.097732
(89917, 2)
ID               0
probability_1    0
dtype: int64


In [114]:
# Run these checks first
print(submission_full.shape)          # should be (89917, 2)
print(submission_full.columns.tolist())  # should be ['ID', 'probability_1']
print(submission_full.isna().sum())   # should be 0 for both
print(submission_full["probability_1"].between(0, 1).all())  # should be True
print(submission_full["probability_1"].describe())  # sanity check the distribution

(89917, 2)
['ID', 'probability_1']
ID               0
probability_1    0
dtype: int64
True
count    89917.000000
mean         0.174252
std          0.143353
min          0.000001
25%          0.068614
50%          0.124955
75%          0.237439
max          0.940998
Name: probability_1, dtype: float64


In [115]:
# ── Add NAICS sector feature ───────────────────────────────────────────────────
# First 2 digits of NAICS = industry sector (e.g. 72 = Food/Hotels, 44 = Retail)

def add_naics_features(df):
    df = df.copy()
    df["naics_sector"] = df["NAICS"].astype(str).str[:2].astype(float)
    df["naics_subsector"] = df["NAICS"].astype(str).str[:3].astype(float)
    return df

# Apply to train dataframe BEFORE the train/val/test split
df = add_naics_features(df)

print("NAICS sector unique values:", df["naics_sector"].nunique())
print("Sample distribution:")
print(df["naics_sector"].value_counts().head(10))

NAICS sector unique values: 25
Sample distribution:
naics_sector
0.0     181851
44.0     76252
81.0     65369
54.0     61357
72.0     60872
23.0     60109
62.0     49686
42.0     43925
45.0     38214
33.0     34402
Name: count, dtype: int64


In [116]:
def add_naics_features(df):
    df = df.copy()
    naics_str = df["NAICS"].astype(str)
    
    # Replace 0 and invalid with NaN, then fill as -1 (unknown sector)
    df["naics_sector"] = naics_str.str[:2].replace("0", "-1").astype(float)
    df["naics_subsector"] = naics_str.str[:3].replace("0", "-1").astype(float)
    
    # Flag for missing NAICS
    df["naics_missing"] = (df["NAICS"] == 0).astype(int)
    
    return df

# Re-apply
df = add_naics_features(df)

print("naics_sector distribution:")
print(df["naics_sector"].value_counts().head(10))
print("\nnaics_missing:", df["naics_missing"].sum(), "rows flagged")

naics_sector distribution:
naics_sector
-1.0     181851
 44.0     76252
 81.0     65369
 54.0     61357
 72.0     60872
 23.0     60109
 62.0     49686
 42.0     43925
 45.0     38214
 33.0     34402
Name: count, dtype: int64

naics_missing: 181851 rows flagged


In [117]:
from sklearn.model_selection import train_test_split

# ── 1. Re-split with new features ─────────────────────────────────────────────
X = df.drop(columns=["MIS_Status"])
y = df["MIS_Status"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# ── 2. Re-encode ───────────────────────────────────────────────────────────────
onehot_cols = ["RevLineCr", "LowDoc"]
woe_cols    = ["State", "BankState"]
target_cols = ["City", "Bank"]

X_train_oh = pd.get_dummies(X_train, columns=onehot_cols, drop_first=True)
X_val_oh   = pd.get_dummies(X_val,   columns=onehot_cols, drop_first=True)
X_test_oh  = pd.get_dummies(X_test,  columns=onehot_cols, drop_first=True)

X_val_oh  = X_val_oh.reindex(columns=X_train_oh.columns, fill_value=0)
X_test_oh = X_test_oh.reindex(columns=X_train_oh.columns, fill_value=0)

target_encoder2 = ce.TargetEncoder(cols=target_cols)
target_encoder2.fit(X_train_oh, y_train)

X_train_te = target_encoder2.transform(X_train_oh)
X_val_te   = target_encoder2.transform(X_val_oh)
X_test_te  = target_encoder2.transform(X_test_oh)

woe_encoder2 = ce.WOEEncoder(cols=woe_cols)
woe_encoder2.fit(X_train_te, y_train)

X_train_enc2 = woe_encoder2.transform(X_train_te)
X_val_enc2   = woe_encoder2.transform(X_val_te)
X_test_enc2  = woe_encoder2.transform(X_test_te)

print("New feature count:", X_train_enc2.shape[1])
print("Any NAs?", X_train_enc2.isna().any().any())

New feature count: 55
Any NAs? False


In [118]:
# ── Retrain H2O GLM on full data with NAICS features ──────────────────────────

# Build full H2O frame
X_full_num2 = X_train_enc2.apply(pd.to_numeric, errors="coerce")
full_h2o2 = h2o.H2OFrame(pd.concat([X_full_num2, y_train.rename("MIS_Status")], axis=1))
full_h2o2["MIS_Status"] = full_h2o2["MIS_Status"].asfactor()

# Force all to numeric
for col in full_h2o2.columns:
    if col == "MIS_Status":
        continue
    if full_h2o2[col].isfactor()[0]:
        full_h2o2[col] = full_h2o2[col].asnumeric()

x_cols2 = [c for c in full_h2o2.columns if c != "MIS_Status"]

# Build val H2O frame
X_val_num2 = X_val_enc2.apply(pd.to_numeric, errors="coerce")
val_h2o2 = h2o.H2OFrame(pd.concat([X_val_num2, y_val.rename("MIS_Status")], axis=1))
val_h2o2["MIS_Status"] = val_h2o2["MIS_Status"].asfactor()

# Match val types to full_h2o2
for col in x_cols2:
    if col not in val_h2o2.columns:
        continue
    if full_h2o2.types[col] != val_h2o2.types[col]:
        if full_h2o2.types[col] == "enum":
            val_h2o2[col] = val_h2o2[col].asfactor()
        else:
            val_h2o2[col] = val_h2o2[col].asnumeric()

# Train
final_glm_naics = H2OGeneralizedLinearEstimator(
    family="binomial",
    alpha=float(best_config["alpha"]),
    lambda_=float(best_config["lambda"]),
    standardize=bool(best_config["standardize"]),
    missing_values_handling=str(best_config["missing_values_handling"]),
    max_iterations=200,
    seed=42
)

final_glm_naics.train(x=x_cols2, y="MIS_Status", training_frame=full_h2o2)
print("Done training!")

# Evaluate on val
val_preds_naics = final_glm_naics.predict(val_h2o2).as_data_frame()["p1"].values
h2o_best_threshold2, _ = find_best_f1_threshold(y_val.values, val_preds_naics, average="macro")
auc, aucpr, ll, f1m, cm = evaluate_model(y_val.values, val_preds_naics, h2o_best_threshold2)
print(f"Val AUCPR (NAICS): {aucpr:.4f}  AUC: {auc:.4f}  Threshold: {h2o_best_threshold2:.2f}")

Done training!


/Users/himavarshakomanduri/Desktop/Spring 2026/BUAN 6341 - Applied Machine Learning/.venv/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


Val AUCPR (NAICS): 0.4461  AUC: 0.7717  Threshold: 0.29


In [119]:
from h2o.estimators.gbm import H2OGradientBoostingEstimator

gbm_model = H2OGradientBoostingEstimator(
    ntrees=200,
    max_depth=6,
    learn_rate=0.05,
    sample_rate=0.8,
    col_sample_rate=0.8,
    seed=42
)

gbm_model.train(x=x_cols2, y="MIS_Status", training_frame=full_h2o2)
print("Done training GBM!")

# Evaluate on val
gbm_val_preds = gbm_model.predict(val_h2o2).as_data_frame()["p1"].values
gbm_threshold, _ = find_best_f1_threshold(y_val.values, gbm_val_preds, average="macro")
auc, aucpr, ll, f1m, cm = evaluate_model(y_val.values, gbm_val_preds, gbm_threshold)
print(f"Val AUCPR (GBM): {aucpr:.4f}  AUC: {auc:.4f}  Threshold: {gbm_threshold:.2f}")

Done training GBM!


/Users/himavarshakomanduri/Desktop/Spring 2026/BUAN 6341 - Applied Machine Learning/.venv/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


Val AUCPR (GBM): 0.5312  AUC: 0.8188  Threshold: 0.31


In [121]:
# Apply NAICS features to holdout
holdout = pd.read_csv("./data/SBA_loans_project_1_holdout_students_valid_no_labels.csv.zip").copy()
holdout = add_naics_features(holdout)

# Fill missing
cat_fill_unknown = ["RevLineCr", "LowDoc", "BankState", "Bank", "City", "State"]
for col in cat_fill_unknown:
    holdout[col] = holdout[col].fillna("Unknown")
holdout["NewExist"] = holdout["NewExist"].fillna(holdout["NewExist"].mode()[0])

# Numeric conversion
num_cols = ["SBA_Appv","GrAppv","DisbursementGross","BalanceGross",
            "CreateJob","RetainedJob","NoEmp","UrbanRural","FranchiseCode"]
for c in num_cols:
    if c in holdout.columns:
        holdout[c] = pd.to_numeric(holdout[c], errors="coerce")

# Feature engineering
holdout["loan_to_sba_ratio"]        = holdout["SBA_Appv"] / (holdout["GrAppv"] + 1)
holdout["disbursement_to_approval"] = holdout["DisbursementGross"] / (holdout["GrAppv"] + 1)
holdout["balance_to_disbursement"]  = holdout["BalanceGross"] / (holdout["DisbursementGross"] + 1)
holdout["jobs_created_ratio"]       = holdout["CreateJob"] / (holdout["NoEmp"] + 1)
holdout["jobs_retained_ratio"]      = holdout["RetainedJob"] / (holdout["NoEmp"] + 1)
holdout["total_jobs"]               = holdout["CreateJob"] + holdout["RetainedJob"]
holdout["jobs_per_employee"]        = (holdout["CreateJob"] + holdout["RetainedJob"]) / (holdout["NoEmp"] + 1)
holdout["is_urban"]                 = (holdout["UrbanRural"] == 1).astype(int)
holdout["is_franchise"]             = (holdout["FranchiseCode"] > 0).astype(int)
holdout["sba_guarantee_diff"]       = holdout["GrAppv"] - holdout["SBA_Appv"]
holdout["log_approval"]             = np.log1p(holdout["GrAppv"])

# Encode
holdout_oh = pd.get_dummies(holdout, columns=["RevLineCr", "LowDoc"], drop_first=True)
holdout_oh = holdout_oh.reindex(columns=X_train_oh.columns, fill_value=0)
holdout_te  = target_encoder2.transform(holdout_oh)
holdout_enc2 = woe_encoder2.transform(holdout_te)
holdout_enc2 = holdout_enc2.reindex(columns=X_train_enc2.columns, fill_value=0)

# H2O frame
holdout_num2 = holdout_enc2.apply(pd.to_numeric, errors="coerce")
holdout_h2o2 = h2o.H2OFrame(holdout_num2)

# Match types to training frame
for col in x_cols2:
    if col not in holdout_h2o2.columns:
        continue
    if full_h2o2.types[col] != holdout_h2o2.types[col]:
        if full_h2o2.types[col] == "enum":
            holdout_h2o2[col] = holdout_h2o2[col].asfactor()
        else:
            holdout_h2o2[col] = holdout_h2o2[col].asnumeric()

# Predict
gbm_holdout_preds = gbm_model.predict(holdout_h2o2).as_data_frame()

# Submission
holdout_ids = pd.read_csv("./data/SBA_loans_project_1_holdout_students_valid_no_labels.csv.zip")["index"]
submission_gbm = pd.DataFrame({
    "ID": holdout_ids.values,
    "probability_1": gbm_holdout_preds["p1"].values
})

submission_gbm.to_csv("submission_gbm.csv", index=False)
print(submission_gbm.head())
print(submission_gbm.shape)
print(submission_gbm.isna().sum())
print(submission_gbm["probability_1"].describe())

/Users/himavarshakomanduri/Desktop/Spring 2026/BUAN 6341 - Applied Machine Learning/.venv/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


   ID  probability_1
0   0       0.069514
1   1       0.031195
2   2       0.258245
3   3       0.184077
4   4       0.038831
(89917, 2)
ID               0
probability_1    0
dtype: int64
count    89917.000000
mean         0.172363
std          0.187455
min          0.001369
25%          0.043005
50%          0.097789
75%          0.233155
max          0.952270
Name: probability_1, dtype: float64
